In [1]:
# Find project files

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

config_files = list(INPUT_ROOT.rglob("taxify_config.json"))
dataset_files = list(INPUT_ROOT.rglob("taxify_documents.jsonl"))
manifest_files = list(INPUT_ROOT.rglob("ingestion_manifest.json"))

print("Configuration files:", config_files)
print("Document datasets:", dataset_files)
print("Ingestion manifests:", manifest_files)

assert config_files, "taxify_config.json was not found. Add Notebook 01 output."
assert dataset_files, "taxify_documents.jsonl was not found. Add Notebook 02 output."
assert manifest_files, "ingestion_manifest.json was not found. Add Notebook 02 output."

CONFIG_PATH = config_files[0]
DATASET_PATH = dataset_files[0]
MANIFEST_PATH = manifest_files[0]

print("\nSelected configuration:", CONFIG_PATH)
print("Selected dataset:", DATASET_PATH)
print("Selected manifest:", MANIFEST_PATH)

Configuration files: [PosixPath('/kaggle/input/notebooks/nallagantisharath/environment-and-project-configuration/taxify/taxify_config.json')]
Document datasets: [PosixPath('/kaggle/input/notebooks/nallagantisharath/irs-document-ingestion/taxify/processed/taxify_documents.jsonl')]
Ingestion manifests: [PosixPath('/kaggle/input/notebooks/nallagantisharath/irs-document-ingestion/taxify/processed/ingestion_manifest.json')]

Selected configuration: /kaggle/input/notebooks/nallagantisharath/environment-and-project-configuration/taxify/taxify_config.json
Selected dataset: /kaggle/input/notebooks/nallagantisharath/irs-document-ingestion/taxify/processed/taxify_documents.jsonl
Selected manifest: /kaggle/input/notebooks/nallagantisharath/irs-document-ingestion/taxify/processed/ingestion_manifest.json


In [2]:
# Verify dependencies and hardware


!pip install -q faiss-cpu sentence-transformers
import json
import platform
import torch
import faiss
import sentence_transformers

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = json.load(file)

with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
    ingestion_manifest = json.load(file)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nProject:", config["project_name"])
print("Embedding model:", config["embedding_model"])
print("Chunk size:", config["chunk_size_words"])
print("Chunk overlap:", config["chunk_overlap_words"])
print("Retrieval top-k:", config["retrieval_top_k"])

print("\nDataset records:", ingestion_manifest["total_records"])
print("Real taxpayer data used:", ingestion_manifest["real_taxpayer_data_used"])

assert ingestion_manifest["total_records"] == 508
assert ingestion_manifest["real_taxpayer_data_used"] is False

print("\nEnvironment and dataset verification passed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.4 MB/s eta 0:00:00
Python: 3.12.13
PyTorch: 2.10.0+cu128
SentenceTransformers: 5.4.1
FAISS: 1.15.1

CUDA available: True
GPU: Tesla T4

Project: Taxify
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Chunk size: 180
Chunk overlap: 35
Retrieval top-k: 4

Dataset records: 508
Real taxpayer data used: False

Environment and dataset verification passed.


In [3]:
# Load and validate the records


import json
from collections import Counter

records = []

with open(DATASET_PATH, "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if line.strip():
            record = json.loads(line)
            record["_line_number"] = line_number
            records.append(record)

print("Loaded records:", len(records))
print("Record fields:", sorted(records[0].keys()))

print("\nClassification counts:")
print(Counter(record.get("classification") for record in records))

print("\nTrust counts:")
print(Counter(record.get("trust") for record in records))

print("\nDocument counts:")
print(Counter(record.get("document_id") for record in records))

empty_text = [
    record["_line_number"]
    for record in records
    if not str(record.get("text", "")).strip()
]

assert len(records) == 508, f"Expected 508 records, found {len(records)}"
assert not empty_text, f"Empty text found on lines: {empty_text[:10]}"

print("\nAll 508 records contain text.")
print("Dataset validation passed.")

Loaded records: 508
Record fields: ['_line_number', 'authorized_roles', 'character_count', 'classification', 'document_id', 'filename', 'page_number', 'source_url', 'text', 'text_sha256', 'trust', 'word_count']

Classification counts:
Counter({'public': 506, 'restricted': 2})

Trust counts:
Counter({'trusted': 506, 'untrusted': 2})

Document counts:
Counter({'p17': 142, 'p550': 117, 'i1065': 70, 'p463': 62, 'p505': 48, 'i1120': 34, 'p501': 31, 'synthetic_restricted_taxpayer': 1, 'synthetic_preparer_memo': 1, 'synthetic_poisoned_tax_note': 1, 'synthetic_unsafe_output': 1})

All 508 records contain text.
Dataset validation passed.


In [4]:
# Create RAG chunks

import hashlib
from collections import Counter

CHUNK_SIZE = int(config["chunk_size_words"])
CHUNK_OVERLAP = int(config["chunk_overlap_words"])
STEP_SIZE = CHUNK_SIZE - CHUNK_OVERLAP

assert CHUNK_SIZE > CHUNK_OVERLAP > 0

def split_into_chunks(text, chunk_size, overlap):
    words = text.split()
    step = chunk_size - overlap
    chunks = []

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]

        if not chunk_words:
            break

        chunks.append({
            "text": " ".join(chunk_words),
            "start_word": start,
            "end_word": start + len(chunk_words),
            "word_count": len(chunk_words)
        })

        if start + chunk_size >= len(words):
            break

    return chunks

chunks = []

for record in records:
    page_chunks = split_into_chunks(
        record["text"],
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    for position, page_chunk in enumerate(page_chunks):
        chunk_id = (
            f'{record["document_id"]}'
            f'_p{record["page_number"]}'
            f'_c{position}'
        )

        chunks.append({
            "chunk_id": chunk_id,
            "document_id": record["document_id"],
            "filename": record["filename"],
            "page_number": record["page_number"],
            "chunk_position": position,
            "text": page_chunk["text"],
            "word_count": page_chunk["word_count"],
            "start_word": page_chunk["start_word"],
            "end_word": page_chunk["end_word"],
            "classification": record["classification"],
            "trust": record["trust"],
            "authorized_roles": record["authorized_roles"],
            "source_url": record["source_url"],
            "text_sha256": hashlib.sha256(
                page_chunk["text"].encode("utf-8")
            ).hexdigest()
        })

print("Original records:", len(records))
print("Generated chunks:", len(chunks))
print("Chunk size:", CHUNK_SIZE)
print("Overlap:", CHUNK_OVERLAP)

print("\nChunks by document:")
print(Counter(chunk["document_id"] for chunk in chunks))

print("\nChunks by classification:")
print(Counter(chunk["classification"] for chunk in chunks))

print("\nChunks by trust:")
print(Counter(chunk["trust"] for chunk in chunks))

assert chunks
assert len({chunk["chunk_id"] for chunk in chunks}) == len(chunks)
assert all(chunk["text"].strip() for chunk in chunks)
assert all(chunk["word_count"] <= CHUNK_SIZE for chunk in chunks)

print("\nChunk validation passed.")


Original records: 508
Generated chunks: 3656
Chunk size: 180
Overlap: 35

Chunks by document:
Counter({'p17': 1207, 'p550': 766, 'i1065': 524, 'p463': 361, 'p505': 291, 'p501': 256, 'i1120': 247, 'synthetic_restricted_taxpayer': 1, 'synthetic_preparer_memo': 1, 'synthetic_poisoned_tax_note': 1, 'synthetic_unsafe_output': 1})

Chunks by classification:
Counter({'public': 3654, 'restricted': 2})

Chunks by trust:
Counter({'trusted': 3654, 'untrusted': 2})

Chunk validation passed.


In [5]:
# Save the chunk dataset

from pathlib import Path
import json
import hashlib

PROJECT_DIR = Path("/kaggle/working/taxify")
VECTOR_DIR = PROJECT_DIR / "vectors"
PROCESSED_DIR = PROJECT_DIR / "processed"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = PROCESSED_DIR / "taxify_chunks.jsonl"

with open(CHUNKS_PATH, "w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

chunks_sha256 = hashlib.sha256(CHUNKS_PATH.read_bytes()).hexdigest()

print("Chunks saved:", CHUNKS_PATH)
print("Saved chunks:", len(chunks))
print("File size:", round(CHUNKS_PATH.stat().st_size / 1024 / 1024, 2), "MB")
print("SHA-256:", chunks_sha256)

assert CHUNKS_PATH.exists()
assert sum(1 for _ in open(CHUNKS_PATH, encoding="utf-8")) == len(chunks)

print("Chunk dataset saved and verified.")

Chunks saved: /kaggle/working/taxify/processed/taxify_chunks.jsonl
Saved chunks: 3656
File size: 5.04 MB
SHA-256: cde92734b4883ed54e51f3158fa5b37b3655e45b3fa0bf6380bfb7365d1efa22
Chunk dataset saved and verified.


In [6]:
# Load the embedding model

import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = config["embedding_model"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading:", EMBEDDING_MODEL_NAME)
print("Device:", DEVICE)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

embedding_model.max_seq_length = 256

embedding_dimension = embedding_model.get_sentence_embedding_dimension()

print("Embedding dimension:", embedding_dimension)
print("Maximum sequence length:", embedding_model.max_seq_length)

assert embedding_dimension == 384

print("Embedding model loaded successfully.")

Loading: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
Maximum sequence length: 256
Embedding model loaded successfully.


/tmp/ipykernel_23/2755349737.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = embedding_model.get_sentence_embedding_dimension()


In [7]:
# Generate and save embeddings

import numpy as np
from datetime import datetime, timezone

chunk_texts = [chunk["text"] for chunk in chunks]

print("Generating embeddings for", len(chunk_texts), "chunks...")

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

EMBEDDINGS_PATH = VECTOR_DIR / "taxify_embeddings.npy"
np.save(EMBEDDINGS_PATH, embeddings)

print("\nEmbedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print(
    "Saved size:",
    round(EMBEDDINGS_PATH.stat().st_size / 1024 / 1024, 2),
    "MB"
)

norms = np.linalg.norm(embeddings, axis=1)

print("Minimum vector norm:", round(float(norms.min()), 6))
print("Maximum vector norm:", round(float(norms.max()), 6))

assert embeddings.shape == (len(chunks), 384)
assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()
assert np.allclose(norms, 1.0, atol=1e-4)

print("Embeddings generated, normalized, and saved successfully.")

Generating embeddings for 3656 chunks...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]


Embedding shape: (3656, 384)
Data type: float32
Saved size: 5.36 MB
Minimum vector norm: 1.0
Maximum vector norm: 1.0
Embeddings generated, normalized, and saved successfully.


In [8]:
# Build the FAISS vector database

import faiss

FAISS_INDEX_PATH = VECTOR_DIR / "taxify_faiss.index"

# Inner-product search on normalized vectors produces cosine similarity.
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(embeddings)

faiss.write_index(faiss_index, str(FAISS_INDEX_PATH))

print("Index type:", type(faiss_index).__name__)
print("Vector dimension:", faiss_index.d)
print("Vectors stored:", faiss_index.ntotal)
print("Index saved:", FAISS_INDEX_PATH)
print(
    "Index size:",
    round(FAISS_INDEX_PATH.stat().st_size / 1024 / 1024, 2),
    "MB"
)

loaded_index = faiss.read_index(str(FAISS_INDEX_PATH))

assert loaded_index.d == 384
assert loaded_index.ntotal == len(chunks)

print("FAISS vector database created and verified successfully.")

Index type: IndexFlatIP
Vector dimension: 384
Vectors stored: 3656
Index saved: /kaggle/working/taxify/vectors/taxify_faiss.index
Index size: 5.36 MB
FAISS vector database created and verified successfully.


In [9]:

# Connect semantic search to FAISS

def normalize_roles(value):
    if isinstance(value, list):
        return [str(role).strip().lower() for role in value]

    return [
        role.strip().lower()
        for role in str(value).split("|")
        if role.strip()
    ]


def search_vector_database(
    query,
    top_k=4,
    role="taxpayer",
    secured=True
):
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve extra candidates so security filters still leave enough results.
    candidate_count = min(max(top_k * 10, 40), faiss_index.ntotal)

    scores, indexes = faiss_index.search(
        query_vector,
        candidate_count
    )

    results = []

    for score, index_position in zip(scores[0], indexes[0]):
        if index_position < 0:
            continue

        chunk = chunks[int(index_position)]
        authorized_roles = normalize_roles(chunk["authorized_roles"])

        if secured:
            if role.lower() not in authorized_roles:
                continue

            if chunk["trust"].lower() != "trusted":
                continue

        results.append({
            "score": round(float(score), 4),
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "filename": chunk["filename"],
            "page_number": chunk["page_number"],
            "classification": chunk["classification"],
            "trust": chunk["trust"],
            "source_url": chunk["source_url"],
            "text": chunk["text"]
        })

        if len(results) == top_k:
            break

    return results


test_query = "What tests determine whether a person is a qualifying child?"

test_results = search_vector_database(
    query=test_query,
    top_k=config["retrieval_top_k"],
    role="taxpayer",
    secured=True
)

print("Query:", test_query)
print("Results:", len(test_results))

for number, result in enumerate(test_results, start=1):
    print(f"\n--- Result {number} ---")
    print("Score:", result["score"])
    print("Document:", result["document_id"])
    print("Page:", result["page_number"])
    print("Classification:", result["classification"])
    print("Trust:", result["trust"])
    print("Text:", result["text"][:500], "...")

Query: What tests determine whether a person is a qualifying child?
Results: 4

--- Result 1 ---
Score: 0.6789
Document: p501
Page: 9
Classification: public
Trust: trusted
Text: the value of your services or those of a member of your household. Qualifying Person See T able 4 to see who is a qualifying person. Any person not described in T able 4 isn’t a qual- ifying person. Example 1 —child. Y our unmarried child lived with you all year and was 18 years old at the end of the year. Y our child didn’t provide more than half of their own support and doesn’t meet the tests to be a qualifying child of anyone else. As a result, this child is your qualifying child (see Qualify ...

--- Result 2 ---
Score: 0.6452
Document: p17
Page: 30
Classification: public
Trust: trusted
Text: next. If a child meets the five tests to be the qualifying child of more than one per- son, there are rules you must use to determine which person can actually treat the child as a qualifying child. See Qualifying Chil

In [10]:
# Save the vector database manifest

import shutil
import hashlib
from datetime import datetime, timezone

OUTPUT_CONFIG_PATH = PROJECT_DIR / "taxify_config.json"
OUTPUT_INGESTION_MANIFEST = PROCESSED_DIR / "ingestion_manifest.json"
VECTOR_MANIFEST_PATH = VECTOR_DIR / "vector_manifest.json"

shutil.copy2(CONFIG_PATH, OUTPUT_CONFIG_PATH)
shutil.copy2(MANIFEST_PATH, OUTPUT_INGESTION_MANIFEST)

def file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

vector_manifest = {
    "project": config["project_name"],
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(embedding_dimension),
    "distance_method": "normalized_inner_product_cosine_similarity",
    "faiss_index_type": type(faiss_index).__name__,
    "chunk_size_words": CHUNK_SIZE,
    "chunk_overlap_words": CHUNK_OVERLAP,
    "source_records": len(records),
    "total_chunks": len(chunks),
    "total_vectors": int(faiss_index.ntotal),
    "real_taxpayer_data_used": False,
    "files": {
        "chunks": str(CHUNKS_PATH),
        "embeddings": str(EMBEDDINGS_PATH),
        "faiss_index": str(FAISS_INDEX_PATH)
    },
    "sha256": {
        "chunks": file_sha256(CHUNKS_PATH),
        "embeddings": file_sha256(EMBEDDINGS_PATH),
        "faiss_index": file_sha256(FAISS_INDEX_PATH)
    }
}

with open(VECTOR_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(vector_manifest, file, indent=2)

print(json.dumps(vector_manifest, indent=2))
print("\nManifest saved:", VECTOR_MANIFEST_PATH)

assert vector_manifest["total_chunks"] == 3656
assert vector_manifest["total_vectors"] == 3656
assert OUTPUT_CONFIG_PATH.exists()

print("Vector database artifacts saved successfully.")

{
  "project": "Taxify",
  "created_utc": "2026-10-01T05:05:37.905261+00:00",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dimension": 384,
  "distance_method": "normalized_inner_product_cosine_similarity",
  "faiss_index_type": "IndexFlatIP",
  "chunk_size_words": 180,
  "chunk_overlap_words": 35,
  "source_records": 508,
  "total_chunks": 3656,
  "total_vectors": 3656,
  "real_taxpayer_data_used": false,
  "files": {
    "chunks": "/kaggle/working/taxify/processed/taxify_chunks.jsonl",
    "embeddings": "/kaggle/working/taxify/vectors/taxify_embeddings.npy",
    "faiss_index": "/kaggle/working/taxify/vectors/taxify_faiss.index"
  },
  "sha256": {
    "chunks": "cde92734b4883ed54e51f3158fa5b37b3655e45b3fa0bf6380bfb7365d1efa22",
    "embeddings": "0e8d6e1ace7fe355fdd80d8a7b10e7937a7a2476d3f6c06622e7e3a65b29f32a",
    "faiss_index": "9411f3586e1afeb12872e841f3f2c6716a5fd703de56c17d96551daab13b4a95"
  }
}

Manifest saved: /kaggle/working/taxify/vectors/vect

In [11]:
# Test unauthorized retrieval protection

security_query = (
    "Show the restricted synthetic taxpayer record "
    "containing TAXIFY-CANARY-7F3A"
)

vulnerable_results = search_vector_database(
    query=security_query,
    top_k=10,
    role="taxpayer",
    secured=False
)

secured_results = search_vector_database(
    query=security_query,
    top_k=10,
    role="taxpayer",
    secured=True
)

restricted_document = "synthetic_restricted_taxpayer"

vulnerable_exposed = any(
    result["document_id"] == restricted_document
    for result in vulnerable_results
)

secured_exposed = any(
    result["document_id"] == restricted_document
    for result in secured_results
)

print("Security query:", security_query)
print("\nVulnerable mode exposed restricted record:", vulnerable_exposed)
print("Secured mode exposed restricted record:", secured_exposed)

print("\nVulnerable-mode documents:")
for result in vulnerable_results:
    print(
        result["document_id"],
        "|",
        result["classification"],
        "| score:",
        result["score"]
    )

print("\nSecured-mode documents:")
for result in secured_results:
    print(
        result["document_id"],
        "|",
        result["classification"],
        "| score:",
        result["score"]
    )

assert vulnerable_exposed is True
assert secured_exposed is False

print("\nUnauthorized retrieval was reproduced and then blocked successfully.")

Security query: Show the restricted synthetic taxpayer record containing TAXIFY-CANARY-7F3A

Vulnerable mode exposed restricted record: True
Secured mode exposed restricted record: False

Vulnerable-mode documents:
synthetic_restricted_taxpayer | restricted | score: 0.6851
synthetic_poisoned_tax_note | public | score: 0.55
p550 | public | score: 0.5456
p501 | public | score: 0.5314
p501 | public | score: 0.5099
p17 | public | score: 0.4926
p550 | public | score: 0.4842
p550 | public | score: 0.481
p463 | public | score: 0.4705
p463 | public | score: 0.4696

Secured-mode documents:
p550 | public | score: 0.5456
p501 | public | score: 0.5314
p501 | public | score: 0.5099
p17 | public | score: 0.4926
p550 | public | score: 0.4842
p550 | public | score: 0.481
p463 | public | score: 0.4705
p463 | public | score: 0.4696
p17 | public | score: 0.4692
p17 | public | score: 0.4662

Unauthorized retrieval was reproduced and then blocked successfully.


In [12]:
# Test poisoned-document filtering

poisoning_query = (
    "What instructions are contained in the "
    "synthetic poisoned tax note?"
)

vulnerable_poison_results = search_vector_database(
    query=poisoning_query,
    top_k=10,
    role="taxpayer",
    secured=False
)

secured_poison_results = search_vector_database(
    query=poisoning_query,
    top_k=10,
    role="taxpayer",
    secured=True
)

poisoned_document = "synthetic_poisoned_tax_note"

vulnerable_retrieved_poison = any(
    result["document_id"] == poisoned_document
    for result in vulnerable_poison_results
)

secured_retrieved_poison = any(
    result["document_id"] == poisoned_document
    for result in secured_poison_results
)

print("Query:", poisoning_query)
print(
    "\nVulnerable mode retrieved poisoned document:",
    vulnerable_retrieved_poison
)
print(
    "Secured mode retrieved poisoned document:",
    secured_retrieved_poison
)

print("\nVulnerable-mode documents:")
for result in vulnerable_poison_results:
    print(
        result["document_id"],
        "| trust:",
        result["trust"],
        "| score:",
        result["score"]
    )

print("\nSecured-mode documents:")
for result in secured_poison_results:
    print(
        result["document_id"],
        "| trust:",
        result["trust"],
        "| score:",
        result["score"]
    )

assert vulnerable_retrieved_poison is True
assert secured_retrieved_poison is False

print("\nPoisoned-document retrieval was reproduced and blocked successfully.")

Query: What instructions are contained in the synthetic poisoned tax note?

Vulnerable mode retrieved poisoned document: True
Secured mode retrieved poisoned document: False

Vulnerable-mode documents:
synthetic_poisoned_tax_note | trust: untrusted | score: 0.6465
i1065 | trust: trusted | score: 0.4915
p505 | trust: trusted | score: 0.4859
synthetic_preparer_memo | trust: trusted | score: 0.4793
p17 | trust: trusted | score: 0.4644
p501 | trust: trusted | score: 0.4622
p463 | trust: trusted | score: 0.4563
p463 | trust: trusted | score: 0.4516
p17 | trust: trusted | score: 0.4507
i1065 | trust: trusted | score: 0.4498

Secured-mode documents:
i1065 | trust: trusted | score: 0.4915
p505 | trust: trusted | score: 0.4859
p17 | trust: trusted | score: 0.4644
p501 | trust: trusted | score: 0.4622
p463 | trust: trusted | score: 0.4563
p463 | trust: trusted | score: 0.4516
p17 | trust: trusted | score: 0.4507
i1065 | trust: trusted | score: 0.4498
p550 | trust: trusted | score: 0.4435
p17 | t

In [13]:
# Save validation evidence

from datetime import datetime, timezone

EVIDENCE_DIR = PROJECT_DIR / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_PATH = EVIDENCE_DIR / "vector_database_validation.json"

def summarize_results(results):
    return [
        {
            "rank": rank,
            "document_id": result["document_id"],
            "page_number": result["page_number"],
            "score": result["score"],
            "classification": result["classification"],
            "trust": result["trust"]
        }
        for rank, result in enumerate(results, start=1)
    ]

validation_evidence = {
    "project": config["project_name"],
    "test_time_utc": datetime.now(timezone.utc).isoformat(),
    "vector_database": "FAISS",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "total_vectors": int(faiss_index.ntotal),
    "tests": [
        {
            "test_id": "VR-001",
            "vulnerability": "unauthorized_retrieval",
            "query": security_query,
            "role": "taxpayer",
            "vulnerable_mode_exposed_restricted_record": vulnerable_exposed,
            "secured_mode_exposed_restricted_record": secured_exposed,
            "vulnerable_results": summarize_results(vulnerable_results),
            "secured_results": summarize_results(secured_results),
            "status": "passed"
        },
        {
            "test_id": "VR-002",
            "vulnerability": "rag_knowledge_base_poisoning",
            "query": poisoning_query,
            "role": "taxpayer",
            "vulnerable_mode_retrieved_poison": vulnerable_retrieved_poison,
            "secured_mode_retrieved_poison": secured_retrieved_poison,
            "vulnerable_results": summarize_results(
                vulnerable_poison_results
            ),
            "secured_results": summarize_results(
                secured_poison_results
            ),
            "status": "passed"
        }
    ],
    "summary": {
        "tests_executed": 2,
        "tests_passed": 2,
        "vulnerabilities_reproduced_in_vulnerable_mode": 2,
        "violations_remaining_in_secured_mode": 0
    }
}

with open(VALIDATION_PATH, "w", encoding="utf-8") as file:
    json.dump(validation_evidence, file, indent=2)

print(json.dumps(validation_evidence["summary"], indent=2))
print("\nEvidence saved:", VALIDATION_PATH)

assert VALIDATION_PATH.exists()
assert validation_evidence["summary"]["tests_passed"] == 2

print("Notebook 03 completed successfully.")

{
  "tests_executed": 2,
  "tests_passed": 2,
  "vulnerabilities_reproduced_in_vulnerable_mode": 2,
  "violations_remaining_in_secured_mode": 0
}

Evidence saved: /kaggle/working/taxify/evidence/vector_database_validation.json
Notebook 03 completed successfully.
